# **Iteso**
# Nicolas Navarro Valenzuela 746812
# Lab03

In [1]:
from spark_utils import SparkUtils
su = SparkUtils("spark://spark-master:7077", "Lab03",)
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 00:14:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,
Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""

column_types = [
    ("Order_ID", "long"),
    ("Company", "string"),
    ("City", "string"),
    ("Customer_Age", "int"),
    ("Order_Value", "float"),
    ("Delivery_Time_Min", "float"),
    ("Distance_Km", "float"),
    ("Items_Count", "float"),
    ("Product_Category", "string"),
    ("Payment_Method", "string"),
    ("Customer_Rating", "float"),
    ("Discount_Applied", "int"),
    ("Delivery_Partner_Rating", "float")
]

qcommerce_schema = SparkUtils.generate_schema(column_types)

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/qcommerce/")

qcommerce_df.printSchema()
print(f"Total records: {qcommerce_df.count()}")
qcommerce_df.show(5)

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: integer (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



Total records: 1000000
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|               1|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|               0|                    3.2|
| 

In [3]:
from pyspark.sql.functions import when, count, isnull

qcommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [4]:
qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns])
qcommerce_null_count_df.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [5]:
qcommerce_null_count_df2 = SparkUtils.null_count(qcommerce_df)
qcommerce_null_count_df2.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [6]:
qcommerce_wo_nulls = qcommerce_df.dropna()
print(f"Total records after dropping nulls: {qcommerce_wo_nulls.count()}")

Total records after dropping nulls: 780992


In [7]:
qcommerce_wo_nulls_fillna = qcommerce_df.fillna({
    "City": "Unknown",
    "Items_Count": 0.0,
    "Customer_Rating": 0.0,
    "Delivery_Partner_Rating": 0.0
})
print(f"Total records after fillna: {qcommerce_wo_nulls_fillna.count()}")

Total records after fillna: 1000000


In [8]:
from pyspark.sql.functions import lit
qcommerce_wo_nulls_fillna = qcommerce_wo_nulls_fillna.withColumn("CodeCountry", lit("IN"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+-----------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|CodeCountry|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+-----------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|               1|                    3.2|         IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|               0|  

In [9]:
from pyspark.sql.functions import col
qcommerce_tax_df = qcommerce_wo_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+-----------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|CodeCountry|        Paid_Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+-----------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|               1|                    3.2|         IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0| 

In [10]:
#TASK 1
Delivery_SLA_df = qcommerce_tax_df.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "Fast") \
                                               .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "On-Time") \
                                                .when(col("Delivery_Time_Min") > 20, "Late")) \
                                                .orderBy(col("Delivery_Time_Min").desc())

Delivery_SLA_df.select("Order_ID", "Company","City", "Delivery_Time_Min","Delivery_SLA").show()

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        Late|
| 1807126|Jio Mart|Haridwar|             40.0|        Late|
| 1165764|Jio Mart|Haridwar|           39.994|        Late|
| 1610720|Jio Mart|Haridwar|           39.994|        Late|
| 1729503|Jio Mart|Haridwar|           39.994|        Late|
| 1951122|Jio Mart|Haridwar|           39.988|        Late|
| 1975896|Jio Mart|Haridwar|           39.988|        Late|
| 1059671|Jio Mart|Haridwar|           39.982|        Late|
| 1594835|Jio Mart|Haridwar|           39.982|        Late|
| 1162804|Jio Mart|Haridwar|           39.982|        Late|
| 1826295|Jio Mart|Haridwar|           39.982|        Late|
| 1961544|Jio Mart|Haridwar|           39.976|        Late|
| 1360875|Jio Mart|Haridwar|           39.964|        Late|
| 1555058|Jio Mart|Haridwar|           3

In [ ]:
#TASK 2
from pyspark.sql.functions import avg, count

Effective_Order_Value_df = qcommerce_tax_df.withColumn("Effective_Order_Value", col("Order_Value") * (1 - (col("Discount_Applied"))))
#Effective_Order_Value_df.select("Order_ID", "Order_Value", "Discount_Applied", "Effective_Order_Value").show(10)

Price_Bucket_df = Effective_Order_Value_df.withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "Low") \
                                               .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "Medium") \
                                                .when(col("Effective_Order_Value") > 500, "High")) \
                                            .groupBy("Price_Bucket").agg( 
                                                                         count("*").alias("Count"), 
                                                                         avg("Effective_Order_Value").alias("Avg_Effective_Order_Value")) \
                                            .orderBy(col("Avg_Effective_Order_Value").desc())
                                                
Price_Bucket_df.show()

+------------+------+-------------------------+
|Price_Bucket| Count|Avg_Effective_Order_Value|
+------------+------+-------------------------+
|        High|268953|        740.4337238893675|
|      Medium|210429|        358.0973233400432|
|         Low|520618|       21.591204157544553|
+------------+------+-------------------------+



In [34]:
#TASK 3
from pyspark.sql.functions import filter

Age_Group_df = qcommerce_tax_df.withColumn("Age_Group", when(col("Customer_Age") < 25, "YOUNG")\
                                                                .when((col("Customer_Age") >= 25) & (col("Customer_Age") < 45), "ADULT")\
                                                                .when(col("Customer_Age") >= 45, "SENIOR")) \
                                        .filter(col("Customer_Age").isNotNull()) \
                                        .filter(col("Customer_Age") > 0) \
                                        .filter(col("Customer_Age") < 100) \
                                        .groupBy("Age_Group", "Product_Category").agg( \
                                                                    count("*").alias("Count"),\
                                                                    avg("Order_Value").alias("Avg_Order_Value")) \
                                        .orderBy(col("Age_Group").asc()) \
                                        .orderBy(col("Count").desc())

Age_Group_df.show()


+---------+-------------------+-----+-----------------+
|Age_Group|   Product_Category|Count|  Avg_Order_Value|
+---------+-------------------+-----+-----------------+
|    ADULT|              Dairy|68512|573.4268899414931|
|    ADULT|      Personal Care|68331| 569.512671998805|
|    ADULT|          Groceries|68291|571.5250993844182|
|    ADULT|          Household|68110|572.9259149188181|
|    ADULT|             Snacks|68100|570.3797974095505|
|    ADULT|          Beverages|67979|572.0329877095578|
|    ADULT|Fruits & Vegetables|67736|569.8593599596651|
|   SENIOR|          Groceries|51030|572.2596391052539|
|   SENIOR|              Dairy|51025| 573.781117268945|
|   SENIOR|             Snacks|50924| 572.687851608881|
|   SENIOR|          Household|50803| 571.172082385334|
|   SENIOR|          Beverages|50746| 568.141013231256|
|   SENIOR|      Personal Care|50707|571.1642560109325|
|   SENIOR|Fruits & Vegetables|50642|573.7244422534909|
|    YOUNG|              Dairy|24235|572.3825174

In [2]:
su._spark.stop()